In [ ]:
import pandas as pd
import numpy as np
from nltk import word_tokenize, sent_tokenize
from pymorphy3 import MorphAnalyzer
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import igraph as ig

In [ ]:
df=pd.read_excel('пользователи из города.xlsx', sheet_name='Sheet1')
df_follower=pd.read_excel('подписки пользователей из города на друг друга.xlsx', sheet_name='Sheet1')

In [ ]:
result=[] #список для сохранения результатов сравнения
all_nodes = pd.concat([df['id']]).unique()
vr = list(all_nodes)
I = ig.Graph(directed=True)

In [ ]:
for ver in vr:
	I.add_vertex(ver)
I.vs["name"]=vr #назначаем ребрам веса

edges = [] #список кортежей ребер
for edge in df_follower.values:
	x=I.vs.find(name=edge[1])
	y=I.vs.find(name=edge[0])
	edges.append((x, y))
I.add_edges(edges)
I.es["weight"]=df_follower["weight"].tolist()
l_weight_visualization = [f"{weight_for_formatting:.2f}" for weight_for_formatting in df_follower["weight"]]

fig, ax = plt.subplots(figsize=(39, 45))
ig.plot(I, vertex_label=I.vs["name"], vertex_size=50, vertex_label_size=16, target=ax, edge_label=l_weight_visualization, edge_label_size=16)
fig.savefig("Граф со связью подпиской.png")

In [ ]:
user_id = []
posts_user_id = []
connection_user = []
posts_connection_user = []
comparsion_tuple_list = []

In [ ]:
from string import punctuation
import re
from nltk.corpus import stopwords
morph = MorphAnalyzer()
from transformers import AutoModel, AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/sbert_large_nlu_ru")
model = AutoModel.from_pretrained("ai-forever/sbert_large_nlu_ru")
stop_w = list(punctuation) + stopwords.words('russian') + ['`', '..', '"', '...', '»', '«', '``', "''", 'это', 'который', 'ещё', '|https', 'либо', 'всё', 'весь', 'свой']

In [ ]:
#предобработка текста
def post_preprocessing(post):
    post_s = sent_tokenize(post) #разделяем текст постов на предложения
    list_post_sent = [] 
    post_w = []
    for sentence in post_s:
        sent_w = word_tokenize(sentence.lower())
        #добавляем слово в список слов, если оно не является стоп-словом или числом
        for s_word in sent_w:
            if (s_word not in stop_w) and not re.match(r'\d+', s_word) and not s_word.startswith('id'):
                normal_w = morph.parse(s_word)[0].normal_form
                post_w.append(normal_w)   
        #если в предложении больше 5 слов
        if len(post_w)>5:
            post_sent = " ".join(post_w)
            #добавляем слова в список предложений
            list_post_sent.append(post_sent)
        post_w = []
    res_post_w = ' '.join(list_post_sent) #строка слов из поста для возврата
    return(res_post_w)

In [ ]:
#пулирование (агрегирование)
def mean_pooling(model_output, attention_mask):
	token_embeddings = model_output[0]
	input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
	sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
	sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
	return sum_embeddings/sum_mask

In [ ]:
#сравнение (фунция вычисления косинусного расстояния)
def comparison(vec_post1, post2):
	prepared_post2 = post_preprocessing(post2)
	if len(prepared_post2)!=0:
		tokenized_post2 = tokenizer(prepared_post2, padding=True, truncation=True, max_length=512, return_tensors='pt')
		with torch.no_grad():
			model_output = model(**tokenized_post2) #получение входа модели
            
		vec_post2 = mean_pooling(model_output,
		tokenized_post2['attention_mask'])[0].detach().numpy()
		dot_product = np.dot(vec_post1, vec_post2)
		vector1_norms = np.linalg.norm(vec_post1)
		vector2_norms = np.linalg.norm(vec_post2)
		cos_distance = dot_product/(vector1_norms*vector2_norms)
	else: cos_distance=0
	return(cos_distance)

In [ ]:
df_size=len(df)
tuple_subscriptions=[(df_follower['follower'][row], df_follower['user_id'][row]) for row in range(df_follower.shape[0])]
for y in range(df_size):
    post1 = df['post'][y]
	prepared_post1 = post_preprocessing(post1)
	tokenized_post1 = tokenizer(prepared_post1, padding=True, truncation=True, max_length=512, return_tensors='pt')
	with torch.no_grad():
        model_output = model(**tokenized_post1)
	vec_post1 = mean_pooling(model_output, tokenized_post1['attention_mask'])[0].detach().numpy()
    
	for z in range(df_size):
        if (df['id'][z]==df['id'][y]) or ((df['id'][z], df['id'][y]) in comparsion_tuple_list) or ((df['id'][z], df['id'][y]) in tuple_subscriptions or (df['id'][y], df['id'][z]) in tuple_subscriptions):
            continue
		else:
            post2=df['post'][z]
			user_id.append(df['id'][y])
			posts_user_id.append(post1)
			connection_user.append(df['id'][z])
			posts_connection_user.append(post2)
			result.append(comparison(vec_post1, post2))
			comparsion_tuple = (df['id'][y], df['id'][z])
			comparsion_tuple_list.append(comparsion_tuple)

In [ ]:
time_df=pd.DataFrame(
    {'user_id': user_id,
	'posts_user': posts_user_id,
	'сonnection_user':  connection_user,
	'posts_сonnection_user': posts_connection_user,
	'comparison_result': result})

In [ ]:
#убираем связи между теми, у кого сходство (косинусное растояние) меньше 0,80
for ed_w in time_df['comparison_result']:
    if (ed_w<0.80): time_df.drop(time_df[time_df['comparison_result']==ed_w].index, inplace=True)

In [ ]:
df_res=pd.DataFrame({
	'src': pd.concat([df_follower['follower'], time_df['user_id'], time_df['сonnection_user']], axis=0),
	'dst': pd.concat([df_follower['user_id'], time_df['сonnection_user'], time_df['user_id']], axis=0),
	'weight':  pd.concat([df_follower['weight'], time_df['comparison_result'], time_df['comparison_result']],  axis=0)})
df_res.to_excel('ребра итогового графа с весами.xlsx', index=False)
time_df.to_excel('результат сравнения.xlsx', index=False)

In [ ]:
ed=pd.DataFrame({'src': pd.concat([time_df['user_id'], time_df['сonnection_user']], axis=0),
                 'dst': pd.concat([time_df['сonnection_user'], time_df['user_id']], axis=0)})

edges = []
for edge in ed.values:
	x=I.vs.find(name=edge[0])
	y=I.vs.find(name=edge[1])
	edges.append((x, y))

I.add_edges(edges)
I.es["weight"]=time_df["comparison_result"].tolist()
l_weight_visualization = [f"{weight_for_formatting:.2f}" for weight_for_formatting in I.es["weight"]]

fig, ax = plt.subplots(figsize=(39, 45))
ig.plot(I, vertex_label=I.vs["name"], vertex_size=50, vertex_label_size=16, target=ax, edge_label=l_weight_visualization, edge_label_size=16)
fig.savefig("Получившийся после сравнения граф.png")